# 02 — EDA Univariate


In [ ]:
import sys
from pathlib import Path
REPO = Path.cwd()
for candidate in [REPO, REPO.parent, REPO.parent.parent]:
    if (candidate / "src" / "neuro").exists():
        REPO = candidate
        break
sys.path.insert(0, str(REPO / "src"))
import os
os.chdir(REPO)
%matplotlib inline


In [ ]:

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import nibabel as nib
from neuro.bids import validate_bids, load_participants
from neuro.features import parse_events
from neuro import viz

report = validate_bids()
runs = report["runs"]
participants = load_participants()
available = runs[runs["bold_exists"]]
print(f"Available runs: {len(available)}")


In [ ]:

# T1w intensity histogram (one subject)
t1_row = available.iloc[0]
t1 = nib.load(t1_row["t1_path"] if t1_row["t1_path"] else available[available["t1_path"].notna()].iloc[0]["t1_path"])
t1_data = t1.get_fdata()
plt.figure(figsize=(6,4))
plt.hist(t1_data[t1_data > 0].ravel(), bins=80, color="#8172B3")
plt.title("T1w voxel intensities (non-zero)")
plt.xlabel("Intensity"); plt.ylabel("Count")
viz.savefig("02_t1w_histogram.png")


In [ ]:

# BOLD mean signal per volume (carpet plot proxy)
bold = nib.load(available.iloc[0]["bold_path"])
data = bold.get_fdata()
mean_ts = data.reshape(-1, data.shape[-1]).mean(axis=0)
plt.figure(figsize=(10,3))
plt.plot(mean_ts, color="#4C72B0")
plt.title(f"Global mean BOLD — {available.iloc[0]['subject']}")
plt.xlabel("Volume"); plt.ylabel("Mean signal")
viz.savefig("02_bold_mean_timeseries.png")


In [ ]:

# Events / stimulus distribution
ev = parse_events(available.iloc[0]["events_path"])
viz.plot_trial_counts(pd.read_csv(available.iloc[0]["events_path"], sep="\t"))
participants.groupby(["group_short","sex"]).size().unstack(fill_value=0)


In [ ]:

from pathlib import Path
nb_name = Path.cwd().name if False else "02_eda_univariate"
!jupyter nbconvert --to html notebooks/02_eda_univariate.ipynb --output-dir notebooks/html 2>/dev/null || true
